In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# SETUP
# ============================================================

# Install required packages
!pip install -q deepl google-genai lingua-language-detector


# ============================================================
# IMPORTS
# ============================================================

import os
import random
import re
from datetime import datetime

import deepl
import numpy as np
import pandas as pd

from google import genai
from google.genai import types

from lingua import Language, LanguageDetectorBuilder
from scipy.stats import wilcoxon


# ============================================================
# API KEY
# ============================================================

# Replace with your own DeepL API key before running the notebook.
DEEPL_KEY = "API_KEY"


# ============================================================
# CLIENT
# ============================================================

deepl_client = deepl.Translator(DEEPL_KEY)

print("API client initialized")


# ============================================================
# PATHS
# ============================================================

TEST_AUGMENTED_PATH = "/content/drive/MyDrive/code_switch_project/data/augmented_dataset/test_augmented.csv"

TRANSLATION_RESULTS_PATH = "/content/drive/MyDrive/code_switch_project/results/translation_results.csv"

TRANSLATION_RESULTS_LLM_PATH = "/content/drive/MyDrive/code_switch_project/results/translation_results_with_llm.csv"

LLM_EXPERIMENT_LOG_PATH = "/content/drive/MyDrive/code_switch_project/results/llm_experiment_log.txt"

BLIND_EVALUATION_PATH = "/content/drive/MyDrive/code_switch_project/results/blind_evaluation.csv"

BLIND_EVALUATION_SCORED_PATH = "/content/drive/MyDrive/code_switch_project/results/blind_evaluation_scored.csv"

EVALUATION_MAPPING_PATH = "/content/drive/MyDrive/code_switch_project/results/evaluation_mapping.csv"

ERROR_ORIGIN_PATH = "/content/drive/MyDrive/code_switch_project/results/error_origin_analysis.csv"

API client initialized


In [ ]:
# ============================================================
# LOAD TEST DATA
# ============================================================

df = pd.read_csv(TEST_AUGMENTED_PATH)

print(f"Dataset size: {len(df):,}")
print("\nLanguage distribution:")
print(df["language"].value_counts())

Dataset size: 39,740

Language distribution:
language
french    15000
german    15000
mixed      9740
Name: count, dtype: int64


In [ ]:
# ============================================================
# SELECT MIXED-LANGUAGE SAMPLE
# ============================================================

np.random.seed(42)

mixed_df = df[df["language"] == "mixed"].copy()

sample_df = mixed_df.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

print(f"Selected sentences: {len(sample_df)}")

sample_df.head()

Selected sentences: 50


,text,language
0,"écoute, vas-y doucement, okay",mixed
1,"oui, mrs reagan",mixed
2,"Tu as rencontré un mec, et jetzt tu veux me pa...",mixed
3,"vielleicht que quand tu auras fini ici, tu pou...",mixed
4,"salut, Cornelia, wie geht's, sowas?",mixed


In [ ]:
# ============================================================
# LANGUAGE DETECTOR
# ============================================================

# Detect French or German for individual words/tokens.
detector = LanguageDetectorBuilder.from_languages(
    Language.FRENCH,
    Language.GERMAN
).build()


def detect_language(word):
    """Return the detected language of a word as FR, DE, or UNK."""

    result = detector.detect_language_of(word)

    if result == Language.FRENCH:
        return "FR"

    if result == Language.GERMAN:
        return "DE"

    return "UNK"

In [ ]:
# ============================================================
# SENTENCE SEGMENTATION
# ============================================================

def segment_sentence(text):
    """
    Split a sentence into consecutive French and German segments
    based on word-level language detection.
    """

    words = text.split()

    segments = []
    current_lang = None
    current_words = []

    for word in words:

        # Remove punctuation while keeping language-specific characters.
        clean = re.sub(
            r"[^\wäöüÄÖÜßéèêàç'-]",
            "",
            word
        )

        if clean == "":
            continue

        lang = detect_language(clean)

        # Start the first segment.
        if current_lang is None:
            current_lang = lang
            current_words.append(word)

        # Continue the current segment if the language is unchanged.
        elif lang == current_lang:
            current_words.append(word)

        # Start a new segment when the detected language changes.
        else:
            segments.append(
                (
                    current_lang,
                    " ".join(current_words)
                )
            )

            current_lang = lang
            current_words = [word]

    # Add the final segment.
    if current_words:
        segments.append(
            (
                current_lang,
                " ".join(current_words)
            )
        )

    return segments

In [ ]:
# ============================================================
# DEEPL TRANSLATION
# ============================================================

def deepl_translate(text):
    """Translate a French segment into German using DeepL."""

    if text.strip() == "":
        return ""

    result = deepl_client.translate_text(
        text,
        source_lang="FR",
        target_lang="DE"
    )

    return result.text

In [ ]:
# ============================================================
# SEGMENTATION, TRANSLATION AND RECOMBINATION
# ============================================================

def segmentation_translation(text):
    """Segment a mixed sentence, translate French parts to German,
    recombine the segments, and refine the final German sentence."""

    segments = segment_sentence(text)

    translated_parts = []

    for lang, chunk in segments:

        if lang == "FR":
            translated_parts.append(
                deepl_translate(chunk)
            )
        else:
            translated_parts.append(chunk)

    # Recombine the translated and original German segments.
    recombined_text = " ".join(translated_parts)

    # Refine the complete German sentence with DeepL Write.
    result = deepl_client.rephrase_text(
        recombined_text,
        target_lang="DE"
    )

    return result.text, segments

In [ ]:
# ============================================================
# DEEPL BASELINE TRANSLATION
# ============================================================

def deepl_baseline(text):
    """Translate the complete mixed sentence directly with DeepL."""

    result = deepl_client.translate_text(
        text,
        source_lang="FR",
        target_lang="DE"
    )

    return result.text

In [ ]:
# ============================================================
# GENERATE TRANSLATION RESULTS
# ============================================================

results = []

for idx, row in sample_df.iterrows():

    text = row["text"]

    print(f"Processing {idx + 1}/{len(sample_df)}")

    # DeepL baseline: translate the complete mixed sentence.
    deepl_out = deepl_baseline(text)

    # Segmentation approach: segment the sentence,
    # translate French segments, and recombine the result.
    segmented_out, _ = segmentation_translation(text)

    # Store all translation outputs.
    results.append({
        "id": idx,
        "original": text,
        "deepl_raw": deepl_out,
        "segmentation_deepl": segmented_out,

        # Filled manually later using the LLM web interface.
        "llm_raw": "",

        "date": datetime.now().strftime("%Y-%m-%d"),
        "deepl_model": "DeepL API"
    })

results_df = pd.DataFrame(results)

results_df.head()

Processing 1/50
Processing 2/50
Processing 3/50
Processing 4/50
Processing 5/50
Processing 6/50
Processing 7/50
Processing 8/50
Processing 9/50
Processing 10/50
Processing 11/50
Processing 12/50
Processing 13/50
Processing 14/50
Processing 15/50
Processing 16/50
Processing 17/50
Processing 18/50
Processing 19/50
Processing 20/50
Processing 21/50
Processing 22/50
Processing 23/50
Processing 24/50
Processing 25/50
Processing 26/50
Processing 27/50
Processing 28/50
Processing 29/50
Processing 30/50
Processing 31/50
Processing 32/50
Processing 33/50
Processing 34/50
Processing 35/50
Processing 36/50
Processing 37/50
Processing 38/50
Processing 39/50
Processing 40/50
Processing 41/50
Processing 42/50
Processing 43/50
Processing 44/50
Processing 45/50
Processing 46/50
Processing 47/50
Processing 48/50
Processing 49/50
Processing 50/50


,id,original,deepl_raw,segmentation_deepl,llm_raw,date,deepl_model
0,0,"- verstanden, wir beginnen maintenant","- Verstanden, wir fangen jetzt an","- verstanden, wir beginnen jetzt",,2026-08-04,DeepL API
1,1,"ja, je sais, et je te promets que je vais vs l...","Ja, ich weiß, und ich verspreche dir, dass ich...","ja, Ich weiß, und ich te versprich mir, dass i...",,2026-08-04,DeepL API
2,2,Roscoe et gagnant sont deux mots qui ne vont p...,"„Roscoe“ und „Sieger“ sind zwei Wörter, die ni...","„Roscoe“ und „Gewinner“ sind zwei Wörter, die ...",,2026-08-04,DeepL API
3,3,"gut, tu sais, c'est très gut, sowas","Gut, weißt du, das ist echt gut, so was","gut, Weißt du, das ist wirklich gut, sowas",,2026-08-04,DeepL API
4,4,"Hunter, arbeite weiter daran, das mot de passe...","Hunter, arbeite weiter daran, das Passwort für...","Hunter, arbeite weiter daran, das Wort de Pass...",,2026-08-04,DeepL API


In [ ]:
# ============================================================
# SAVE TRANSLATION RESULTS
# ============================================================

results_df.to_csv(
    TRANSLATION_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Translation results saved.")

Saved to Drive.


In [ ]:
# ============================================================
# VERIFY LLM TRANSLATION RESULTS
# ============================================================

df = pd.read_csv(TRANSLATION_RESULTS_LLM_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 LLM translations:")
print(df["llm_raw"].head())

empty_llm = (df["llm_raw"].fillna("") == "").sum()
print(f"\nEmpty LLM translations: {empty_llm}")

Dataset shape: (50, 7)

Columns:
['id', 'original', 'deepl_raw', 'segmentation_deepl', 'llm_raw', 'date', 'deepl_model']

First 5 LLM translations:
0                       Verstanden, wir beginnen jetzt
1    Ja, ich weiß, und ich verspreche dir, dass ich...
2    Roscoe und Gewinner sind zwei Wörter, die nich...
3               Gut, weißt du, das ist sehr gut, sowas
4    Hunter, arbeite weiter daran, das Passwort für...
Name: llm_raw, dtype: object

Empty LLM translations: 0


In [ ]:
# ============================================================
# SAVE LLM EXPERIMENT LOG
# ============================================================

log_text = """
LLM Translation Baseline Experiment

Model:
Gemini 3.6 Flash

Interface:
Gemini Web Interface

Date:
2026-08-04

Prompt:
Translate this message into German.

Only output the translation.

Procedure:
50 French-German chat messages were translated independently.
Each sentence was processed in a separate conversation to avoid
context influence.
Raw outputs were saved without modification.

Generation parameters:
Temperature and other generation parameters were not configurable
through the web interface.

Notes:
Unexpected outputs were retained unchanged as raw model outputs.
"""

with open(LLM_EXPERIMENT_LOG_PATH, "w", encoding="utf-8") as f:
    f.write(log_text)

print(f"LLM experiment log saved: {LLM_EXPERIMENT_LOG_PATH}")

Log file created.


In [ ]:
# ============================================================
# CREATE BLIND EVALUATION AND HIDDEN METHOD MAPPING
# ============================================================

# Load the translation results.
df = pd.read_csv(TRANSLATION_RESULTS_LLM_PATH)

blind_results = []
mapping_results = []

for _, row in df.iterrows():

    # Collect the three translation methods.
    translations = [
        ("DeepL", row["deepl_raw"]),
        ("Segmentation+DeepL", row["segmentation_deepl"]),
        ("Gemini", row["llm_raw"])
    ]

    # Randomly shuffle the methods to blind the manual evaluation.
    random.shuffle(translations)

    # Store the shuffled translations without revealing their methods.
    blind_results.append({
        "id": row["id"],
        "original": row["original"],
        "A": translations[0][1],
        "B": translations[1][1],
        "C": translations[2][1],

        # Filled manually during evaluation.
        "score_A": "",
        "score_B": "",
        "score_C": ""
    })

    # Store the hidden mapping between positions and methods.
    mapping_results.append({
        "id": row["id"],
        "A_method": translations[0][0],
        "B_method": translations[1][0],
        "C_method": translations[2][0]
    })

# Save the blind evaluation file.
blind_df = pd.DataFrame(blind_results)

blind_df.to_csv(
    BLIND_EVALUATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save the hidden method mapping.
mapping_df = pd.DataFrame(mapping_results)

mapping_df.to_csv(
    EVALUATION_MAPPING_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Blind evaluation file saved: {BLIND_EVALUATION_PATH}")
print(f"Hidden mapping saved: {EVALUATION_MAPPING_PATH}")

print("\nPreview:")
blind_df.head()

Blind evaluation file saved:
/content/drive/MyDrive/code_switch_project/results/blind_evaluation.csv

Hidden mapping saved:
/content/drive/MyDrive/code_switch_project/results/evaluation_mapping.csv

Preview:


,id,original,A,B,C,score_A,score_B,score_C
0,0,"- verstanden, wir beginnen maintenant","- verstanden, wir beginnen jetzt","Verstanden, wir beginnen jetzt","- Verstanden, wir fangen jetzt an",,,
1,1,"ja, je sais, et je te promets que je vais vs l...","Ja, ich weiß, und ich verspreche dir, dass ich...","ja, Ich weiß, und ich te versprich mir, dass i...","Ja, ich weiß, und ich verspreche dir, dass ich...",,,
2,2,Roscoe et gagnant sont deux mots qui ne vont p...,"„Roscoe“ und „Sieger“ sind zwei Wörter, die ni...","Roscoe und Gewinner sind zwei Wörter, die nich...","„Roscoe“ und „Gewinner“ sind zwei Wörter, die ...",,,
3,3,"gut, tu sais, c'est très gut, sowas","Gut, weißt du, das ist echt gut, so was","Gut, weißt du, das ist sehr gut, sowas","gut, Weißt du, das ist wirklich gut, sowas",,,
4,4,"Hunter, arbeite weiter daran, das mot de passe...","Hunter, arbeite weiter daran, das Wort de Pass...","Hunter, arbeite weiter daran, das Passwort für...","Hunter, arbeite weiter daran, das Passwort für...",,,


In [ ]:
# ============================================================
# VERIFY BLIND EVALUATION SCORES
# ============================================================

df_eval = pd.read_csv(BLIND_EVALUATION_SCORED_PATH)

print("Dataset shape:", df_eval.shape)

print("\nColumns:")
print(df_eval.columns.tolist())

print("\nMissing scores:")
print(
    df_eval[["score_A", "score_B", "score_C"]].isna().sum()
)

Dataset shape: (50, 8)

Columns:
['id', 'original', 'A', 'B', 'C', 'score_A', 'score_B', 'score_C']

Missing scores:
score_A    0
score_B    0
score_C    0
dtype: int64


In [ ]:
# ============================================================
# UNBLIND EVALUATION RESULTS
# ============================================================

# Load the scored blind evaluation and hidden method mapping.
evaluation_df = pd.read_csv(BLIND_EVALUATION_SCORED_PATH)
mapping_df = pd.read_csv(EVALUATION_MAPPING_PATH)

# Match each blind score with its corresponding translation method.
results_unblinded = evaluation_df.merge(
    mapping_df,
    on="id",
    how="inner"
)

print("Rows:", len(results_unblinded))

print("\nColumns:")
print(results_unblinded.columns.tolist())

print("\nFirst 5 unblinded results:")
print(
    results_unblinded[
        [
            "id",
            "score_A",
            "score_B",
            "score_C",
            "A_method",
            "B_method",
            "C_method"
        ]
    ].head()
)

Rows: 50

Columns:
['id', 'original', 'A', 'B', 'C', 'score_A', 'score_B', 'score_C', 'A_method', 'B_method', 'C_method']

First 5 unblinded results:
   id  score_A  score_B  score_C            A_method            B_method  \
0   0        5        5        5  Segmentation+DeepL              Gemini   
1   1        5        1        5               DeepL  Segmentation+DeepL   
2   2        5        5        2               DeepL              Gemini   
3   3        5        5        5               DeepL              Gemini   
4   4        2        5        5  Segmentation+DeepL              Gemini   

             C_method  
0               DeepL  
1              Gemini  
2  Segmentation+DeepL  
3  Segmentation+DeepL  
4               DeepL  


In [ ]:
# ============================================================
# ASSIGN SCORES TO METHODS
# ============================================================

# Create columns for the score of each translation method.
results_unblinded["DeepL_score"] = None
results_unblinded["Segmentation_DeepL_score"] = None
results_unblinded["Gemini_score"] = None

# Assign each blind score to its corresponding method.
for idx, row in results_unblinded.iterrows():

    for position in ["A", "B", "C"]:

        method = row[f"{position}_method"]
        score = row[f"score_{position}"]

        if method == "DeepL":
            results_unblinded.at[idx, "DeepL_score"] = score

        elif method == "Segmentation+DeepL":
            results_unblinded.at[idx, "Segmentation_DeepL_score"] = score

        elif method == "Gemini":
            results_unblinded.at[idx, "Gemini_score"] = score

# Preview method-specific scores.
print(
    results_unblinded[
        [
            "id",
            "DeepL_score",
            "Segmentation_DeepL_score",
            "Gemini_score"
        ]
    ].head(10)
)

   id DeepL_score Segmentation_DeepL_score Gemini_score
0   0           5                        5            5
1   1           5                        1            5
2   2           5                        2            5
3   3           5                        5            5
4   4           5                        2            5
5   5           5                        4            5
6   6           5                        1            5
7   7           5                        2            5
8   8           4                        2            5
9   9           5                        4            5


In [ ]:
# ============================================================
# SUMMARY STATISTICS BY TRANSLATION METHOD
# ============================================================

# Extract and convert all method-specific scores to numeric values.
method_scores = results_unblinded[
    [
        "DeepL_score",
        "Segmentation_DeepL_score",
        "Gemini_score"
    ]
].astype(float)

# Calculate descriptive statistics for each method.
summary = pd.DataFrame({
    "Method": [
        "DeepL",
        "Segmentation + DeepL",
        "Gemini"
    ],
    "Mean": [
        method_scores["DeepL_score"].mean(),
        method_scores["Segmentation_DeepL_score"].mean(),
        method_scores["Gemini_score"].mean()
    ],
    "Median": [
        method_scores["DeepL_score"].median(),
        method_scores["Segmentation_DeepL_score"].median(),
        method_scores["Gemini_score"].median()
    ],
    "Std": [
        method_scores["DeepL_score"].std(),
        method_scores["Segmentation_DeepL_score"].std(),
        method_scores["Gemini_score"].std()
    ],
    "Min": [
        method_scores["DeepL_score"].min(),
        method_scores["Segmentation_DeepL_score"].min(),
        method_scores["Gemini_score"].min()
    ],
    "Max": [
        method_scores["DeepL_score"].max(),
        method_scores["Segmentation_DeepL_score"].max(),
        method_scores["Gemini_score"].max()
    ]
})

print(summary.to_string(index=False))

              Method  Mean  Median      Std  Min  Max
               DeepL  4.76     5.0 0.624663  3.0  5.0
Segmentation + DeepL  3.14     3.0 1.370387  1.0  5.0
              Gemini  4.66     5.0 0.894655  1.0  5.0


In [ ]:
# ============================================================
# SENTENCE-LEVEL SCORE DIFFERENCES
# ============================================================

# Calculate the score difference for each sentence.
# Negative values mean the comparison method scored lower than DeepL.
results_unblinded["Segmentation_vs_DeepL"] = (
    results_unblinded["Segmentation_DeepL_score"]
    - results_unblinded["DeepL_score"]
)

results_unblinded["Gemini_vs_DeepL"] = (
    results_unblinded["Gemini_score"]
    - results_unblinded["DeepL_score"]
)

# Show the frequency of each paired difference.
print("Segmentation + DeepL vs DeepL:")
print(
    results_unblinded["Segmentation_vs_DeepL"]
    .value_counts()
    .sort_index()
)

print("\nGemini vs DeepL:")
print(
    results_unblinded["Gemini_vs_DeepL"]
    .value_counts()
    .sort_index()
)

# Show the average paired difference.
print("\nAverage differences:")

print(
    "Segmentation + DeepL:",
    results_unblinded["Segmentation_vs_DeepL"].mean()
)

print(
    "Gemini:",
    results_unblinded["Gemini_vs_DeepL"].mean()
)

Segmentation + DeepL vs DeepL:
Segmentation_vs_DeepL
-4     6
-3    12
-2     9
-1     8
0     11
1      3
2      1
Name: count, dtype: int64

Gemini vs DeepL:
Gemini_vs_DeepL
-4     1
-2     2
-1     5
0     37
1      2
2      3
Name: count, dtype: int64

Average differences:
Segmentation + DeepL: -1.62
Gemini: -0.1


In [ ]:
# ============================================================
# WILCOXON SIGNED-RANK TESTS
# ============================================================

# Extract paired scores for the three translation methods.
deepl = results_unblinded["DeepL_score"].astype(float)
segmented = results_unblinded["Segmentation_DeepL_score"].astype(float)
gemini = results_unblinded["Gemini_score"].astype(float)

# Compare each method with the DeepL baseline.
# The same sentences are used for both methods in each comparison.
seg_vs_deepl = wilcoxon(
    segmented,
    deepl,
    alternative="two-sided"
)

gemini_vs_deepl = wilcoxon(
    gemini,
    deepl,
    alternative="two-sided"
)

# Display test results.
print("Segmentation + DeepL vs DeepL")
print("Wilcoxon statistic:", seg_vs_deepl.statistic)
print("p-value:", seg_vs_deepl.pvalue)

print("\nGemini vs DeepL")
print("Wilcoxon statistic:", gemini_vs_deepl.statistic)
print("p-value:", gemini_vs_deepl.pvalue)

Segmentation + DeepL vs DeepL
Wilcoxon statistic: 34.5
p-value: 5.609768025027787e-07

Gemini vs DeepL
Wilcoxon statistic: 38.0
p-value: 0.5914457018346772


In [ ]:
# ============================================================
# CREATE ERROR-ORIGIN ANALYSIS TABLE
# ============================================================

# Load the existing translation results.
# No new translations are generated.
translation_df = pd.read_csv(TRANSLATION_RESULTS_LLM_PATH)

# Select the 35 sentences where Segmentation + DeepL
# performed worse than the DeepL baseline.
error_origin_df = results_unblinded[
    results_unblinded["Segmentation_DeepL_score"]
    < results_unblinded["DeepL_score"]
].copy()

# Add only the existing Segmentation + DeepL translation.
# The original sentence is already present in results_unblinded.
error_origin_df = error_origin_df.merge(
    translation_df[["id", "segmentation_deepl"]],
    on="id",
    how="left"
)

# Recreate the detected language segments for inspection.
# This performs segmentation only; no translation is generated.
error_origin_df["segments"] = error_origin_df["original"].apply(
    lambda text: " | ".join(
        f"{lang}: {chunk}"
        for lang, chunk in segment_sentence(text)
    )
)

# Create an empty column for manual error classification.
error_origin_df["error_origin"] = ""

# Keep only the information needed for the analysis.
error_origin_df = error_origin_df[
    [
        "id",
        "original",
        "DeepL_score",
        "Segmentation_DeepL_score",
        "segments",
        "segmentation_deepl",
        "error_origin"
    ]
]

# Save the separate analysis file.
error_origin_df.to_csv(
    ERROR_ORIGIN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Cases to analyze: {len(error_origin_df)}")
print(f"Saved to: {ERROR_ORIGIN_PATH}")

error_origin_df.head()

Cases to analyze: 35
Saved to: /content/drive/MyDrive/code_switch_project/results/error_origin_analysis.csv


,id,original,DeepL_score,Segmentation_DeepL_score,segments,segmentation_deepl,error_origin
0,1,"ja, je sais, et je te promets que je vais vs l...",5,1,"DE: ja, | FR: je sais, et je | DE: te | FR: pr...","ja, Ich weiß, und ich te versprich mir, dass i...",
1,2,Roscoe et gagnant sont deux mots qui ne vont p...,5,2,FR: Roscoe et gagnant sont deux mots qui | DE:...,"„Roscoe“ und „Gewinner“ sind zwei Wörter, die ...",
2,4,"Hunter, arbeite weiter daran, das mot de passe...",5,2,"DE: Hunter, arbeite weiter daran, das | FR: mo...","Hunter, arbeite weiter daran, das Wort de Pass...",
3,5,ich soll einen englisch-aufsatz bis demain sch...,5,4,DE: ich soll einen englisch-aufsatz bis | FR: ...,ich soll einen englisch-aufsatz bis morgen sch...,
4,6,je vais m'étendre dans mon büro.,5,1,FR: je vais m'étendre dans mon | DE: büro.,Ich werde mich in meinem büro.,


In [ ]:
# ============================================================
# ERROR-ORIGIN AND ERROR-SEVERITY ANALYSIS
# ============================================================

# Load the completed manual error-origin analysis.
error_origin_df = pd.read_csv(ERROR_ORIGIN_PATH)

# ------------------------------------------------------------
# 1. Distribution of Segmentation + DeepL scores
# ------------------------------------------------------------

score_distribution = (
    error_origin_df["Segmentation_DeepL_score"]
    .astype(float)
    .value_counts()
    .sort_index(ascending=False)
)

print("Segmentation + DeepL score distribution:")
print(score_distribution)

print("\nPercentages:")
print(
    (score_distribution / len(error_origin_df) * 100)
    .round(1)
)


# ------------------------------------------------------------
# 2. Error-origin distribution
# ------------------------------------------------------------

# Exclude empty/None entries from the error-origin analysis.
# These cases remain in the dataset but are not treated as
# identifiable pipeline errors.
origin_df = error_origin_df[
    error_origin_df["error_origin"].fillna("").str.strip() != ""
].copy()

origin_counts = (
    origin_df["error_origin"]
    .value_counts()
)

print("\nError-origin distribution:")
print(origin_counts)

print("\nPercentages among identifiable errors:")
print(
    (origin_counts / len(origin_df) * 100)
    .round(1)
)


# ------------------------------------------------------------
# 3. Score severity by error origin
# ------------------------------------------------------------

severity_by_origin = pd.crosstab(
    origin_df["error_origin"],
    origin_df["Segmentation_DeepL_score"]
)

print("\nSegmentation + DeepL scores by error origin:")
print(severity_by_origin)

Segmentation + DeepL score distribution:
Segmentation_DeepL_score
4.0     9
3.0     6
2.0    14
1.0     6
Name: count, dtype: int64

Percentages:
Segmentation_DeepL_score
4.0    25.7
3.0    17.1
2.0    40.0
1.0    17.1
Name: count, dtype: float64

Error-origin distribution:
error_origin
Segmentation error                                      22
Translation/Recombination error                          5
segmentation error + Translation/Recombination error     2
Name: count, dtype: int64

Percentages among identifiable errors:
error_origin
Segmentation error                                      75.9
Translation/Recombination error                         17.2
segmentation error + Translation/Recombination error     6.9
Name: count, dtype: float64

Segmentation + DeepL scores by error origin:
Segmentation_DeepL_score                            1   2  3  4
error_origin                                                   
Segmentation error                                  3  12  4  3
Transla